# Restaurant Manager – Table, Server & Seat Functions

This notebook implements four core operations for the `cps714.restaurant_manager` schema:
1. **Create restaurant tables** (with auto-generated seats)
2. **Assign tables to servers** (via a `server_table_assignment` mapping table)
3. **Assign food to seat numbers** (orders placed at individual seats)
4. **Clear table** (remove orders, release server, reset to available)

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

SCHEMA = "cps714.restaurant_manager"

# Create the server_table_assignment mapping table if it doesn't already exist.
# We check first (instead of CREATE OR REPLACE) so we don't wipe existing assignment data.
_existing = spark.sql(f"SHOW TABLES IN {SCHEMA} LIKE 'server_table_assignment'").collect()
if len(_existing) == 0:
    spark.sql(f"""
        CREATE TABLE {SCHEMA}.server_table_assignment (
            assignment_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
            table_id      BIGINT,
            employee_id   BIGINT,
            assigned_at   TIMESTAMP,
            status        STRING
        )
    """)
    print(f"Created {SCHEMA}.server_table_assignment.")
else:
    print(f"{SCHEMA}.server_table_assignment already exists — skipping creation.")

In [0]:
def create_restaurant_table(restaurant_id, table_number, capacity=4):
    """
    Create a new restaurant table and its associated seats.
    
    Parameters:
        restaurant_id (int): The restaurant this table belongs to.
        table_number   (int): The human-visible table number.
        capacity       (int): Number of seats at the table (default 4).
    
    Returns:
        int: The new table_id.
    """
    # Validate restaurant exists
    restaurant = spark.sql(f"SELECT 1 FROM {SCHEMA}.restaurant WHERE restaurant_id = {restaurant_id}")
    if restaurant.count() == 0:
        raise ValueError(f"Restaurant {restaurant_id} does not exist.")
    
    # Check for duplicate table_number in the same restaurant
    existing = spark.sql(f"""
        SELECT 1 FROM {SCHEMA}.restaurant_table 
        WHERE restaurant_id = {restaurant_id} AND table_number = {table_number}
    """)
    if existing.count() > 0:
        raise ValueError(f"Table number {table_number} already exists in restaurant {restaurant_id}.")
    
    # Insert the new table row (table_id is GENERATED ALWAYS AS IDENTITY — omit it)
    spark.sql(f"""
        INSERT INTO {SCHEMA}.restaurant_table (restaurant_id, table_number, capacity, status)
        VALUES ({restaurant_id}, {table_number}, {capacity}, 'available')
    """)
    
    # Retrieve the auto-generated table_id
    new_table_id = spark.sql(f"""
        SELECT table_id FROM {SCHEMA}.restaurant_table 
        WHERE restaurant_id = {restaurant_id} AND table_number = {table_number}
        ORDER BY table_id DESC LIMIT 1
    """).collect()[0]["table_id"]
    
    # Generate seats for this table (seat_id is GENERATED ALWAYS AS IDENTITY — omit it)
    seat_values = ", ".join(f"({new_table_id}, {i})" for i in range(1, capacity + 1))
    spark.sql(f"""
        INSERT INTO {SCHEMA}.seat (table_id, seat_number)
        VALUES {seat_values}
    """)
    
    print(f"Created table {table_number} (id={new_table_id}) with {capacity} seats in restaurant {restaurant_id}.")
    return new_table_id

In [0]:
def assign_table_to_server(table_id, employee_id):
    """
    Assign (or reassign) a restaurant table to a server (employee with role='server').
    
    Parameters:
        table_id    (int): The table to assign.
        employee_id (int): The server (employee) to assign it to.
    
    Returns:
        int: The assignment_id.
    """
    # Validate table exists
    table = spark.sql(f"SELECT * FROM {SCHEMA}.restaurant_table WHERE table_id = {table_id}")
    if table.count() == 0:
        raise ValueError(f"Table {table_id} does not exist.")
    
    # Validate employee exists and is a server
    emp = spark.sql(f"""
        SELECT * FROM {SCHEMA}.employee 
        WHERE employee_id = {employee_id} AND role = 'server' AND status = 'active'
    """)
    if emp.count() == 0:
        raise ValueError(f"Employee {employee_id} is not an active server.")
    
    # Check if table already has an active assignment; if so, deactivate it
    existing = spark.sql(f"""
        SELECT assignment_id FROM {SCHEMA}.server_table_assignment 
        WHERE table_id = {table_id} AND status = 'active'
    """)
    if existing.count() > 0:
        spark.sql(f"""
            UPDATE {SCHEMA}.server_table_assignment 
            SET status = 'inactive' 
            WHERE table_id = {table_id} AND status = 'active'
        """)
        print(f"Deactivated previous assignment for table {table_id}.")
    
    # Insert new assignment (assignment_id is GENERATED ALWAYS AS IDENTITY — omit it)
    spark.sql(f"""
        INSERT INTO {SCHEMA}.server_table_assignment (table_id, employee_id, assigned_at, status)
        VALUES ({table_id}, {employee_id}, current_timestamp(), 'active')
    """)
    
    # Retrieve the auto-generated assignment_id
    new_assignment_id = spark.sql(f"""
        SELECT assignment_id FROM {SCHEMA}.server_table_assignment 
        WHERE table_id = {table_id} AND status = 'active'
        ORDER BY assignment_id DESC LIMIT 1
    """).collect()[0]["assignment_id"]
    
    # Update table status to 'occupied'
    spark.sql(f"""
        UPDATE {SCHEMA}.restaurant_table 
        SET status = 'occupied' 
        WHERE table_id = {table_id}
    """)
    
    # Get server name for the message
    emp_row = emp.collect()[0]
    server_name = emp_row["first_name"] + " " + emp_row["last_name"]
    print(f"Assigned table {table_id} to server {server_name} (employee_id={employee_id}).")
    return new_assignment_id

In [0]:
def assign_food_to_seat(seat_id, menu_item_id, quantity=1):
    """
    Assign a food item (menu item) to a specific seat at a table.
    
    Parameters:
        seat_id      (int): The seat receiving the order.
        menu_item_id (int): The menu item being ordered.
        quantity     (int): How many of this item (default 1).
    
    Returns:
        int: The seat_order_id.
    """
    # Validate seat exists
    seat = spark.sql(f"SELECT * FROM {SCHEMA}.seat WHERE seat_id = {seat_id}")
    if seat.count() == 0:
        raise ValueError(f"Seat {seat_id} does not exist.")
    
    # Validate menu item exists and is available
    item = spark.sql(f"""
        SELECT * FROM {SCHEMA}.menu_item 
        WHERE menu_item_id = {menu_item_id} AND available = true
    """)
    if item.count() == 0:
        raise ValueError(f"Menu item {menu_item_id} does not exist or is unavailable.")
    
    # Get item name and seat info for the message
    item_row = item.collect()[0]
    seat_row = seat.collect()[0]
    item_name = item_row["name"]
    
    # Insert the order (seat_order_id is GENERATED ALWAYS AS IDENTITY — omit it)
    spark.sql(f"""
        INSERT INTO {SCHEMA}.seat_order_item (seat_id, menu_item_id, quantity, order_time, status)
        VALUES ({seat_id}, {menu_item_id}, {quantity}, current_timestamp(), 'ordered')
    """)
    
    # Retrieve the auto-generated seat_order_id
    new_order_id = spark.sql(f"""
        SELECT seat_order_id FROM {SCHEMA}.seat_order_item 
        WHERE seat_id = {seat_id} AND menu_item_id = {menu_item_id}
        ORDER BY seat_order_id DESC LIMIT 1
    """).collect()[0]["seat_order_id"]
    
    print(f"Assigned {quantity}x {item_name} (menu_item_id={menu_item_id}) to seat {seat_id} (seat_number={seat_row['seat_number']}, table_id={seat_row['table_id']}).")
    return new_order_id

In [0]:
def clear_table(table_id):
    """
    Clear a restaurant table: remove all food orders for its seats,
    deactivate the active server assignment, and reset the table to 'available'.
    
    Parameters:
        table_id (int): The table to clear.
    
    Returns:
        bool: True if the table was cleared.
    """
    # Validate table exists
    table = spark.sql(f"SELECT * FROM {SCHEMA}.restaurant_table WHERE table_id = {table_id}")
    if table.count() == 0:
        raise ValueError(f"Table {table_id} does not exist.")
    
    table_row = table.collect()[0]
    table_number = table_row["table_number"]
    
    # 1) Count and delete all seat_order_item records for seats at this table
    order_count = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {SCHEMA}.seat_order_item soi
        JOIN {SCHEMA}.seat s ON soi.seat_id = s.seat_id
        WHERE s.table_id = {table_id}
    """).collect()[0]["cnt"]
    
    if order_count > 0:
        spark.sql(f"""
            DELETE FROM {SCHEMA}.seat_order_item
            WHERE seat_id IN (
                SELECT seat_id FROM {SCHEMA}.seat WHERE table_id = {table_id}
            )
        """)
        print(f"Cleared {order_count} order(s) from table {table_number}.")
    else:
        print(f"No existing orders to clear for table {table_number}.")
    
    # 2) Deactivate the active server assignment (if any)
    active_assignment = spark.sql(f"""
        SELECT assignment_id FROM {SCHEMA}.server_table_assignment 
        WHERE table_id = {table_id} AND status = 'active'
    """)
    if active_assignment.count() > 0:
        spark.sql(f"""
            UPDATE {SCHEMA}.server_table_assignment 
            SET status = 'cleared' 
            WHERE table_id = {table_id} AND status = 'active'
        """)
        print(f"Deactivated server assignment for table {table_number}.")
    else:
        print(f"No active server assignment to deactivate for table {table_number}.")
    
    # 3) Reset the table status to 'available'
    spark.sql(f"""
        UPDATE {SCHEMA}.restaurant_table 
        SET status = 'available' 
        WHERE table_id = {table_id}
    """)
    print(f"Table {table_number} (id={table_id}) is now available.")
    return True

In [0]:
# ── Demo / Test ──────────────────────────────────────────────

# 1) Create a new table (table #7, capacity 4) in restaurant 1
new_table = create_restaurant_table(restaurant_id=1, table_number=7, capacity=4)

# Show the new table and its seats
print("\nNew table:")
spark.sql(f"SELECT * FROM {SCHEMA}.restaurant_table WHERE table_id = {new_table}").show()
print("Seats for new table:")
spark.sql(f"SELECT * FROM {SCHEMA}.seat WHERE table_id = {new_table} ORDER BY seat_number").show()

# 2) Assign the new table to a server (employee_id=3 is Carla Patel, a server)
assign_table_to_server(table_id=new_table, employee_id=3)

# Show the assignment
print("Server-table assignments:")
spark.sql("""
    SELECT a.assignment_id, a.table_id, a.employee_id, 
           e.first_name, e.last_name, a.status, a.assigned_at
    FROM cps714.restaurant_manager.server_table_assignment a
    JOIN cps714.restaurant_manager.employee e ON a.employee_id = e.employee_id
    ORDER BY a.assignment_id DESC
""").show(truncate=False)

# 3) Assign food to seats at the new table
seat_ids = spark.sql(f"SELECT seat_id FROM {SCHEMA}.seat WHERE table_id = {new_table} ORDER BY seat_number").collect()
first_seat = seat_ids[0]["seat_id"]
second_seat = seat_ids[1]["seat_id"]

# Order Bruschetta (menu_item_id=1) for seat 1 and Margherita Pizza (menu_item_id=4) for seat 2
assign_food_to_seat(seat_id=first_seat, menu_item_id=1, quantity=2)
assign_food_to_seat(seat_id=second_seat, menu_item_id=4, quantity=1)

# Show the orders
print("Seat orders:")
spark.sql("""
    SELECT soi.seat_order_id, soi.seat_id, s.seat_number, s.table_id,
           mi.name AS item, soi.quantity, soi.order_time, soi.status
    FROM cps714.restaurant_manager.seat_order_item soi
    JOIN cps714.restaurant_manager.seat s ON soi.seat_id = s.seat_id
    JOIN cps714.restaurant_manager.menu_item mi ON soi.menu_item_id = mi.menu_item_id
    ORDER BY soi.seat_order_id
""").show(truncate=False)

In [0]:
# ════════════════════════════════════════════════════════════
# FULL FUNCTIONALITY DEMONSTRATION
# ════════════════════════════════════════════════════════════
# This cell demonstrates all three functions end-to-end:
#   1. create_restaurant_table  — create a table + auto-generate seats
#   2. assign_table_to_server   — assign a server to the table
#   3. assign_food_to_seat      — place food orders at individual seats

print("=" * 60)
print("  RESTAURANT MANAGER — FULL DEMONSTRATION")
print("=" * 60)

# ── 1) CREATE A NEW RESTAURANT TABLE ────────────────────────
print("\n▶ STEP 1: Creating a new restaurant table")
print("-" * 60)

# Pick the next available table number (find max + 1)
next_table_num = spark.sql(f"""
    SELECT COALESCE(MAX(table_number), 0) + 1 AS next_num
    FROM {SCHEMA}.restaurant_table
    WHERE restaurant_id = 1
""").collect()[0]["next_num"]

demo_table = create_restaurant_table(
    restaurant_id=1,
    table_number=next_table_num,
    capacity=4
)

# Show the new table record
print(f"\nTable record (table_id={demo_table}):")
spark.sql(f"""
    SELECT table_id, table_number, capacity, status
    FROM {SCHEMA}.restaurant_table
    WHERE table_id = {demo_table}
""").show()

# Show the auto-generated seats
print(f"Auto-generated seats for table_id={demo_table}:")
demo_seats = spark.sql(f"""
    SELECT seat_id, table_id, seat_number
    FROM {SCHEMA}.seat
    WHERE table_id = {demo_table}
    ORDER BY seat_number
""")
demo_seats.show()

# Collect seat IDs for ordering later
seat_rows = demo_seats.collect()

# ── 2) ASSIGN THE TABLE TO A SERVER ────────────────────────
print("\n▶ STEP 2: Assigning the table to a server")
print("-" * 60)

# Find an available server who isn't already assigned to a table
available_servers = spark.sql(f"""
    SELECT e.employee_id, e.first_name, e.last_name
    FROM {SCHEMA}.employee e
    LEFT JOIN {SCHEMA}.server_table_assignment a
      ON e.employee_id = a.employee_id AND a.status = 'active'
    WHERE e.role = 'server' AND e.status = 'active'
      AND a.assignment_id IS NULL
    ORDER BY e.employee_id
    LIMIT 1
""").collect()

if available_servers:
    demo_server = available_servers[0]
    server_id = demo_server["employee_id"]
    server_full_name = f"{demo_server['first_name']} {demo_server['last_name']}"
    print(f"Selected available server: {server_full_name} (id={server_id})")
else:
    # Fallback: use any active server
    server_id = 4  # David Kim
    server_full_name = "David Kim"
    print(f"No unassigned servers found; using: {server_full_name} (id={server_id})")

assign_table_to_server(table_id=demo_table, employee_id=server_id)

# Show the active assignment
print(f"\nActive server-table assignments:")
spark.sql(f"""
    SELECT a.assignment_id, a.table_id, t.table_number,
           e.employee_id, e.first_name, e.last_name, e.role,
           a.status, a.assigned_at
    FROM {SCHEMA}.server_table_assignment a
    JOIN {SCHEMA}.restaurant_table t ON a.table_id = t.table_id
    JOIN {SCHEMA}.employee e ON a.employee_id = e.employee_id
    WHERE a.status = 'active'
    ORDER BY a.assignment_id DESC
""").show(truncate=False)

# ── 3) ASSIGN FOOD TO SEAT NUMBERS ─────────────────────────
print("\n▶ STEP 3: Assigning food to individual seats")
print("-" * 60)

# Show available menu items
print("Available menu items:")
spark.sql(f"""
    SELECT menu_item_id, name, price, category
    FROM {SCHEMA}.menu_item
    WHERE restaurant_id = 1 AND available = true
    ORDER BY category, menu_item_id
""").show(truncate=False)

# Order a variety of items across all 4 seats
orders = [
    (seat_rows[0]["seat_id"], 1,  2),   # Seat 1: 2x Bruschetta (Appetizer)
    (seat_rows[1]["seat_id"], 4,  1),   # Seat 2: 1x Margherita Pizza (Main)
    (seat_rows[2]["seat_id"], 5,  1),   # Seat 3: 1x Spaghetti Carbonara (Main)
    (seat_rows[3]["seat_id"], 13, 3),   # Seat 4: 3x Tiramisu (Dessert)
]

for seat_id, menu_item_id, qty in orders:
    assign_food_to_seat(seat_id=seat_id, menu_item_id=menu_item_id, quantity=qty)

# ── FINAL SUMMARY ──────────────────────────────────────────
print("\n" + "=" * 60)
print("  FINAL SUMMARY — TABLE", next_table_num)
print("=" * 60)

print("\nTable & Server:")
spark.sql(f"""
    SELECT t.table_number, t.capacity, t.status AS table_status,
           concat(e.first_name, ' ', e.last_name) AS server_name,
           a.status AS assignment_status
    FROM {SCHEMA}.restaurant_table t
    LEFT JOIN {SCHEMA}.server_table_assignment a
      ON t.table_id = a.table_id AND a.status = 'active'
    LEFT JOIN {SCHEMA}.employee e ON a.employee_id = e.employee_id
    WHERE t.table_id = {demo_table}
""").show(truncate=False)

print("Seat Orders:")
spark.sql(f"""
    SELECT s.seat_number,
           mi.name AS item,
           mi.category,
           mi.price,
           soi.quantity,
           (mi.price * soi.quantity) AS line_total,
           soi.status AS order_status,
           soi.order_time
    FROM {SCHEMA}.seat_order_item soi
    JOIN {SCHEMA}.seat s ON soi.seat_id = s.seat_id
    JOIN {SCHEMA}.menu_item mi ON soi.menu_item_id = mi.menu_item_id
    WHERE s.table_id = {demo_table}
    ORDER BY s.seat_number
""").show(truncate=False)

# Show the bill total
print("Bill Total:")
spark.sql(f"""
    SELECT SUM(mi.price * soi.quantity) AS grand_total
    FROM {SCHEMA}.seat_order_item soi
    JOIN {SCHEMA}.seat s ON soi.seat_id = s.seat_id
    JOIN {SCHEMA}.menu_item mi ON soi.menu_item_id = mi.menu_item_id
    WHERE s.table_id = {demo_table}
""").show()

In [0]:
# ════════════════════════════════════════════════════════════
# VERIFY DATA IS PERSISTED IN CATALOG TABLES
# ════════════════════════════════════════════════════════════

print("=" * 60)
print("  CATALOG TABLE VERIFICATION")
print("=" * 60)

# ── restaurant_table ──
print("\n▸ restaurant_table (all rows):")
spark.sql(f"""
    SELECT table_id, restaurant_id, table_number, capacity, status
    FROM {SCHEMA}.restaurant_table
    ORDER BY table_id
""").show(truncate=False)

# ── seat ──
print("▸ seat (rows for demo tables):")
spark.sql(f"""
    SELECT s.seat_id, s.table_id, s.seat_number, t.table_number
    FROM {SCHEMA}.seat s
    JOIN {SCHEMA}.restaurant_table t ON s.table_id = t.table_id
    WHERE s.table_id >= 7
    ORDER BY s.table_id, s.seat_number
""").show(truncate=False)

# ── server_table_assignment ──
print("▸ server_table_assignment (all rows):")
spark.sql(f"""
    SELECT a.assignment_id, a.table_id, t.table_number,
           e.first_name, e.last_name, a.status, a.assigned_at
    FROM {SCHEMA}.server_table_assignment a
    JOIN {SCHEMA}.restaurant_table t ON a.table_id = t.table_id
    JOIN {SCHEMA}.employee e ON a.employee_id = e.employee_id
    ORDER BY a.assignment_id
""").show(truncate=False)

# ── seat_order_item ──
print("▸ seat_order_item (all orders):")
spark.sql(f"""
    SELECT soi.seat_order_id, s.table_id, t.table_number, s.seat_number,
           mi.name AS item, soi.quantity, soi.order_time, soi.status
    FROM {SCHEMA}.seat_order_item soi
    JOIN {SCHEMA}.seat s ON soi.seat_id = s.seat_id
    JOIN {SCHEMA}.restaurant_table t ON s.table_id = t.table_id
    JOIN {SCHEMA}.menu_item mi ON soi.menu_item_id = mi.menu_item_id
    ORDER BY soi.seat_order_id
""").show(truncate=False)

# ── Summary counts ──
print("▸ Summary counts:")
spark.sql(f"""
    SELECT
      (SELECT COUNT(*) FROM {SCHEMA}.restaurant_table) AS total_tables,
      (SELECT COUNT(*) FROM {SCHEMA}.seat) AS total_seats,
      (SELECT COUNT(*) FROM {SCHEMA}.server_table_assignment) AS total_assignments,
      (SELECT COUNT(*) FROM {SCHEMA}.seat_order_item) AS total_orders
""").show()

In [0]:
# ════════════════════════════════════════════════════════════
# DEMO: clear_table FUNCTION
# ════════════════════════════════════════════════════════════
# Demonstrates clearing a table: removing orders, releasing the server,
# and resetting the table status to 'available'.

print("=" * 60)
print("  CLEAR TABLE DEMONSTRATION")
print("=" * 60)

# Pick the most recently created table (has orders + server assignment)
clear_target = spark.sql(f"""
    SELECT table_id, table_number, status
    FROM {SCHEMA}.restaurant_table
    WHERE status = 'occupied'
    ORDER BY table_id DESC
    LIMIT 1
""").collect()[0]

clear_table_id = clear_target["table_id"]
clear_table_num = clear_target["table_number"]

print(f"\nTarget: table {clear_table_num} (id={clear_table_id}), status={clear_target['status']}")

# Show what's on the table before clearing
print("\n▶ BEFORE clearing:")
print("-" * 60)
print("Orders on this table:")
spark.sql(f"""
    SELECT soi.seat_order_id, s.seat_number, mi.name AS item, 
           soi.quantity, soi.status AS order_status
    FROM {SCHEMA}.seat_order_item soi
    JOIN {SCHEMA}.seat s ON soi.seat_id = s.seat_id
    JOIN {SCHEMA}.menu_item mi ON soi.menu_item_id = mi.menu_item_id
    WHERE s.table_id = {clear_table_id}
    ORDER BY s.seat_number
""").show(truncate=False)

print("Server assignment for this table:")
spark.sql(f"""
    SELECT a.assignment_id, e.first_name, e.last_name, a.status
    FROM {SCHEMA}.server_table_assignment a
    JOIN {SCHEMA}.employee e ON a.employee_id = e.employee_id
    WHERE a.table_id = {clear_table_id}
    ORDER BY a.assignment_id
""").show(truncate=False)

# ── Call clear_table ──
print("\n▶ CLEARING TABLE...")
print("-" * 60)
clear_table(table_id=clear_table_id)

# Show the result after clearing
print("\n▶ AFTER clearing:")
print("-" * 60)
print("Table status:")
spark.sql(f"""
    SELECT table_id, table_number, status
    FROM {SCHEMA}.restaurant_table
    WHERE table_id = {clear_table_id}
""").show(truncate=False)

print("Orders remaining on this table (should be empty):")
remaining = spark.sql(f"""
    SELECT COUNT(*) AS remaining_orders
    FROM {SCHEMA}.seat_order_item soi
    JOIN {SCHEMA}.seat s ON soi.seat_id = s.seat_id
    WHERE s.table_id = {clear_table_id}
""").collect()[0]["remaining_orders"]
print(f"  remaining_orders = {remaining}")

print("Server assignments for this table (should show 'cleared'):")
spark.sql(f"""
    SELECT a.assignment_id, e.first_name, e.last_name, a.status
    FROM {SCHEMA}.server_table_assignment a
    JOIN {SCHEMA}.employee e ON a.employee_id = e.employee_id
    WHERE a.table_id = {clear_table_id}
    ORDER BY a.assignment_id
""").show(truncate=False)